In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# Market Validation Workbook

This notebook is the first market-validation workbook for the swaption project.
The current focus is curve validation using a U.S. Treasury yield-curve proxy snapshot,
with a representative sample swaption placed on top of that curve.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.data_loader import load_curve_points_csv, load_swaption_spec_csv
from swaption_pricing.market_validation import curve_node_rows, discount_factor_rows, load_json_metadata, trade_summary


In [ ]:
curve_csv = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/curve_points.csv'
metadata_json = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/ust_yield_curve_snapshot.json'
spec_csv = PROJECT_ROOT / 'data/raw/example/swaption_spec.csv'

curve = load_curve_points_csv(curve_csv)
spec = load_swaption_spec_csv(spec_csv)
metadata = load_json_metadata(metadata_json)

curve_df = pd.DataFrame(curve_node_rows(curve))
discount_df = pd.DataFrame(discount_factor_rows(curve))
trade_df = pd.DataFrame([trade_summary(curve, spec, black_vol=0.20)])

metadata

## Raw Curve Snapshot

This section shows the normalized curve nodes derived from the public U.S. Treasury yield-curve proxy source.

In [ ]:
curve_df

In [ ]:
discount_df

In [ ]:
ax = curve_df.plot(x='maturity', y='zero_rate', marker='o', figsize=(8, 4), title='U.S. Treasury Yield Curve Proxy')
ax.set_ylabel('Zero Rate')
plt.show()

ax = discount_df.plot(x='maturity', y='discount_factor', marker='o', figsize=(8, 4), title='Discount Factor Curve')
ax.set_ylabel('Discount Factor')
plt.show()

## Representative Swaption on the Proxy Curve

This is not yet a fully vendor-calibrated swaption market snapshot. It is a market-validation bridge:

- the curve is market-connected through a public proxy source
- the sample trade remains a representative research trade
- this allows the pricing stack to be evaluated on top of a real observed rate environment

In [ ]:
trade_df

## Interpretation

- This curve is a public-data U.S. Treasury proxy rather than a full dealer-grade SOFR swap curve.
- It is still useful for demonstrating how the pricing stack responds to a real observed rates regime.
- In the next validation stage, the project should add ATM swaption volatility and, if available, one real or manually curated smile slice.